In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#### Constants

In [ ]:
str_bucket = 'assessment-alt'
str_task = '01_eda'
str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)
print(f'Bucket: {str_bucket}')
print(f'Task: {str_task}')

#### Load Data

In [ ]:
%%time
df = pd.read_csv(f's3://{str_bucket}/00_data_collection/transaction_data.csv', index_col=0)
print(f'Shape: {df.shape}')
df.head()

#### Shape and Dtypes

In [ ]:
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'\nDtypes:')
print(df.dtypes)

#### Missing Values

In [ ]:
sr_propna = df.isnull().mean().sort_values(ascending=True)
print(sr_propna)

fig, ax = plt.subplots(figsize=(10, 5))
sr_propna.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Proportion Missing by Column')
ax.set_xlabel('Proportion Missing')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/propna.png', dpi=150, bbox_inches='tight')
plt.show()

#### Target Distribution

In [ ]:
print(df['price'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=100, color='steelblue', edgecolor='black')
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(np.log1p(df['price']), bins=100, color='steelblue', edgecolor='black')
axes[1].set_title('Log(Price + 1) Distribution')
axes[1].set_xlabel('Log(Price + 1)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{str_dirname_output}/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

#### Grade Distribution

In [ ]:
sr_grade = df['grade'].value_counts().sort_index()
print(sr_grade)

fig, ax = plt.subplots(figsize=(10, 5))
sr_grade.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Grade Distribution')
ax.set_xlabel('Grade')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

#### Grading Company Distribution

In [ ]:
sr_grading = df['grading_company'].value_counts()
print(sr_grading)

fig, ax = plt.subplots(figsize=(8, 5))
sr_grading.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Grading Company Distribution')
ax.set_xlabel('Grading Company')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/grading_company_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

#### Top Subjects

In [ ]:
sr_subject = df['subject'].str.lower().value_counts().head(20)
print(sr_subject)

fig, ax = plt.subplots(figsize=(10, 8))
sr_subject.iloc[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Top 20 Subjects by Transaction Count')
ax.set_xlabel('Transaction Count')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/top_subjects.png', dpi=150, bbox_inches='tight')
plt.show()

#### Top Brands

In [ ]:
sr_brand = df['brand'].str.lower().value_counts().head(20)
print(sr_brand)

fig, ax = plt.subplots(figsize=(10, 8))
sr_brand.iloc[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Top 20 Brands by Transaction Count')
ax.set_xlabel('Transaction Count')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/top_brands.png', dpi=150, bbox_inches='tight')
plt.show()

#### Variety Analysis

In [ ]:
int_null_variety = df['variety'].isnull().sum()
int_non_null_variety = df['variety'].notna().sum()
print(f'Null variety (likely base cards): {int_null_variety:,} ({int_null_variety/len(df)*100:.1f}%)')
print(f'Non-null variety: {int_non_null_variety:,} ({int_non_null_variety/len(df)*100:.1f}%)')

sr_variety = df['variety'].str.lower().value_counts().head(20)
print(f'\nTop 20 varieties:')
print(sr_variety)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(['Null (Base)', 'Has Variety'], [int_null_variety, int_non_null_variety],
            color='steelblue', edgecolor='black')
axes[0].set_title('Variety: Null vs Non-Null')
axes[0].set_ylabel('Count')

sr_variety.iloc[::-1].plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Top 20 Varieties')
axes[1].set_xlabel('Transaction Count')

plt.tight_layout()
plt.savefig(f'{str_dirname_output}/variety_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

#### Temporal Analysis

In [ ]:
df['date'] = pd.to_datetime(df['date'])
sr_monthly = df.set_index('date').resample('M').size()

fig, ax = plt.subplots(figsize=(14, 5))
sr_monthly.plot(ax=ax, color='steelblue', linewidth=2)
ax.set_title('Transactions Over Time (Monthly)')
ax.set_xlabel('Date')
ax.set_ylabel('Transaction Count')
ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.7, label='COVID-19')
ax.legend()
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/transactions_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

#### Price by Grade

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
df.boxplot(column='price', by='grade', ax=ax, showfliers=False)
ax.set_yscale('log')
ax.set_title('Price Distribution by Grade (Log Scale)')
ax.set_xlabel('Grade')
ax.set_ylabel('Price ($, Log Scale)')
plt.suptitle('')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/price_by_grade.png', dpi=150, bbox_inches='tight')
plt.show()

#### Price by Grading Company

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
df.boxplot(column='price', by='grading_company', ax=ax, showfliers=False)
ax.set_yscale('log')
ax.set_title('Price Distribution by Grading Company (Log Scale)')
ax.set_xlabel('Grading Company')
ax.set_ylabel('Price ($, Log Scale)')
plt.suptitle('')
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/price_by_grading_company.png', dpi=150, bbox_inches='tight')
plt.show()

#### Text Normalization Issues

In [ ]:
# Check for case inconsistencies in subject
sr_subject_raw = df['subject'].value_counts()
sr_subject_lower = df['subject'].str.lower().value_counts()
print(f'Unique subjects (raw): {sr_subject_raw.shape[0]}')
print(f'Unique subjects (lowered): {sr_subject_lower.shape[0]}')
print(f'Duplicates from case: {sr_subject_raw.shape[0] - sr_subject_lower.shape[0]}')

# Show examples
df_tmp = df.groupby(df['subject'].str.lower())['subject'].nunique()
df_tmp = df_tmp[df_tmp > 1].sort_values(ascending=False)
print(f'\nSubjects with multiple case variants: {len(df_tmp)}')
print(df_tmp.head(10))

# Check brands
sr_brand_raw = df['brand'].value_counts()
sr_brand_lower = df['brand'].str.lower().value_counts()
print(f'\nUnique brands (raw): {sr_brand_raw.shape[0]}')
print(f'Unique brands (lowered): {sr_brand_lower.shape[0]}')
print(f'Duplicates from case: {sr_brand_raw.shape[0] - sr_brand_lower.shape[0]}')

#### Price Over Time by Top Subjects

In [ ]:
list_str_top_subjects = df['subject'].str.lower().value_counts().head(5).index.tolist()
df_top = df[df['subject'].str.lower().isin(list_str_top_subjects)].copy()
df_top['subject_lower'] = df_top['subject'].str.lower()
df_top['month'] = df_top['date'].dt.to_period('M')

fig, ax = plt.subplots(figsize=(14, 6))
for str_subject in list_str_top_subjects:
    df_subj = df_top[df_top['subject_lower'] == str_subject]
    sr_monthly_median = df_subj.groupby('month')['price'].median()
    sr_monthly_median.index = sr_monthly_median.index.to_timestamp()
    ax.plot(sr_monthly_median.index, sr_monthly_median.values, label=str_subject, linewidth=2)

ax.set_title('Median Price Over Time - Top 5 Subjects')
ax.set_xlabel('Date')
ax.set_ylabel('Median Price ($)')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.savefig(f'{str_dirname_output}/price_over_time_top_subjects.png', dpi=150, bbox_inches='tight')
plt.show()

#### Takeaways

- **100K transactions** for 2018 basketball cards, spanning May 2018 to Feb 2022
- **Price is heavily right-skewed** — median $95, mean $460, max $4M. Log-transform will be essential for modeling
- **Grade 10 dominates** (~51% of transactions), followed by grade 9 (~36%). Very few low grades.
- **PSA is the dominant grader** (87%), with BGS at 13% and SGC negligible
- **~45% of variety is null** — these are likely base cards. Variety is a key price differentiator.
- **Text normalization needed** — subject and brand have case inconsistencies that need resolving before creating card keys
- **Temporal patterns** — transaction volume increased significantly post-COVID (2020+), with a peak around early 2021
- **Price varies significantly by grade** — grade 10 commands a large premium over grade 9
- **Top subjects** (Luka Doncic, Trae Young, etc.) have many more transactions than the long tail of players